# KAP-FinQA-TR — RAG Pipeline & Sonuç Görselleri
**Notebook 4/4:** RAG pipeline, standart benchmark tablosu ve görselleştirmeler

Bu notebook önce RAG pipeline'ını çalıştırır, ardından Drive'daki tüm JSON sonuçlarını  
okuyarak rapordaki standart tabloyu ve grafikleri üretir.

**Ön koşul:** Notebook 1, 2 ve 3 tamamlanmış ve sonuçlar Drive'a kaydedilmiş olmalı.

In [ ]:
# HÜCRE 1 — Kurulum
!pip install -q transformers peft bitsandbytes trl datasets \
             accelerate rouge-score nltk chromadb \
             sentence-transformers huggingface_hub pymupdf matplotlib

In [ ]:
# HÜCRE 2 — Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
os.chdir('/content/drive/MyDrive/kap_finqa')
shutil.copy('/content/drive/MyDrive/kap_finqa/4_rag_pipeline.py', '/content/4_rag_pipeline.py')
print('✅ Hazır')

## RAG Pipeline (Katman 2)

In [ ]:
# HÜCRE 3 — RAG: Qwen2.5-1.5B + ChromaDB (219 PDF)
# Corpus: BIST100 faaliyet raporları, 1.417 chunk, multilingual-e5-small embedding
!python /content/4_rag_pipeline.py \
    --model Qwen2.5-1.5B \
    --pdf_dir /content/drive/MyDrive/kap_finqa/faaliyet_raporlari \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --k 3 5

In [ ]:
# HÜCRE 4 — RAG sonuçlarını Drive'a kaydet
import shutil
shutil.copy('/content/rag_results_Qwen2.5-1.5B.json',
            '/content/drive/MyDrive/kap_finqa/rag_results_Qwen2.5-1.5B.json')
print('✅ RAG sonuçları kaydedildi')

## Standart Benchmark Sonuç Tablosu

In [ ]:
# HÜCRE 5 — Tüm JSON sonuçlarını oku, tablo oluştur
import json, os, numpy as np, pandas as pd

DRIVE = '/content/drive/MyDrive/kap_finqa'

def load_json(fname):
    path = f'{DRIVE}/{fname}'
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

models = ['SmolLM2-360M', 'TinyLlama-1.1B', 'Qwen2.5-1.5B', 'Gemma-4-E2B']
rows = []
for m in models:
    zs  = load_json(f'zero_shot_{m}.json')
    ft  = load_json(f'eval_finetuned_{m}.json')
    row = {'Model': m}
    row['ZS ROUGE-1']  = round(float(zs.get('rouge1', 0)), 4)  if zs else '—'
    row['FT ROUGE-1']  = round(float(ft.get('rouge1', 0)), 4)  if ft else '—'
    row['FT ROUGE-L']  = round(float(ft.get('rougeL', 0)), 4)  if ft else '—'
    row['FT BLEU']     = round(float(ft.get('bleu', 0)), 4)    if ft else '—'
    row['Inf.(ms)']    = ft.get('inference_ms', '—')           if ft else '—'
    row['Disk(MB)']    = ft.get('model_size_mb', '—')          if ft else '—'
    row['VRAM(GB)']    = ft.get('vram_gb', '—')                if ft else '—'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df.to_csv(f'{DRIVE}/benchmark_results.csv')
print('\n✅ benchmark_results.csv kaydedildi')

## Görselleştirmeler

In [ ]:
# HÜCRE 6 — ZS vs FT ROUGE-1 karşılaştırma grafiği
import matplotlib.pyplot as plt
import numpy as np

models_order = ['SmolLM2-360M', 'TinyLlama-1.1B', 'Qwen2.5-1.5B', 'Gemma-4-E2B']
zs_scores = [float(df.loc[m, 'ZS ROUGE-1']) if df.loc[m, 'ZS ROUGE-1'] != '—' else 0 for m in models_order]
ft_scores = [float(df.loc[m, 'FT ROUGE-1']) if df.loc[m, 'FT ROUGE-1'] != '—' else 0 for m in models_order]

x = np.arange(len(models_order))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, zs_scores, width, label='Zero-Shot', color='#AEC6E8', edgecolor='white')
bars2 = ax.bar(x + width/2, ft_scores, width, label='Fine-Tuned (QLoRA)', color='#1F4E79', edgecolor='white')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('ROUGE-1 F1', fontsize=12)
ax.set_title('KAP-FinQA-TR — Zero-Shot vs QLoRA Fine-Tuned ROUGE-1', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_order, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 0.35)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)

for bar in bars1 + bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.004, f'{h:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{DRIVE}/fig_zs_vs_ft_rouge1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik kaydedildi')

In [ ]:
# HÜCRE 7 — Parametre başına verimlilik (ROUGE-1 / 100M param)
param_m = {'SmolLM2-360M': 205, 'TinyLlama-1.1B': 617, 'Qwen2.5-1.5B': 890, 'Gemma-4-E2B': 2493}
efficiency = {m: (float(df.loc[m,'FT ROUGE-1']) / (param_m[m]/100)) 
              if df.loc[m,'FT ROUGE-1'] != '—' else 0 for m in models_order}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ECC71', '#3498DB', '#9B59B6', '#E74C3C']
bars = ax.bar(models_order, [efficiency[m] for m in models_order], color=colors, edgecolor='white')
ax.set_ylabel('ROUGE-1 / 100M Parametre', fontsize=11)
ax.set_title('Parametre Başına Verimlilik (Edge Deployment Skoru)', fontsize=12, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 0.0003, f'{h:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(f'{DRIVE}/fig_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Verimlilik grafiği kaydedildi')